# Week 10 Lab Work

**Dataset:** Adult / Census Income (UCI Machine Learning Repository)

1. Load the same pre-processed dataset as the last lab works.
2. Split it into training (70%), testing (15%), and validation (15%).
3. Apply the k-means algorithm.
4. Compare the results with the previous results (Week 07 Decision Trees, Week 09 SVM and KNN).

In [1]:
import os
import urllib.request

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, silhouette_score

cols = ["age", "workclass", "fnlwgt", "education", "education-num",
        "marital-status", "occupation", "relationship", "race", "sex",
        "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]


def get_file(data_filename):
    for base in [".", ".."]:
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if data_filename in files:
                return os.path.join(root, data_filename)

    print(f"'{data_filename}' not found in the workspace. Downloading it dynamically...")
    url = f"https://archive.ics.uci.edu/ml/machine-learning-databases/adult/{data_filename}"
    urllib.request.urlretrieve(url, data_filename)
    return data_filename


# 1. Load and pre-process the data
path = get_file("adult.data")
print(f"Loading data from: {path}")
df = pd.read_csv(path, header=None, names=cols, skipinitialspace=True, na_values="?")
df = df.dropna().drop_duplicates().reset_index(drop=True)
df["income"] = (df["income"] == ">50K").astype(int)
df = pd.get_dummies(df, drop_first=True)

print("Shape:", df.shape)
print("Missing cells:", df.isnull().sum().sum())
df.head()

Loading data from: ./adult.data
Shape: (30139, 97)
Missing cells: 0


,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,income,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,...,native-country_Portugal,native-country_Puerto-Rico,native-country_Scotland,native-country_South,native-country_Taiwan,native-country_Thailand,native-country_Trinadad&Tobago,native-country_United-States,native-country_Vietnam,native-country_Yugoslavia
0,39,77516,13,2174,0,40,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,50,83311,13,0,0,13,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,38,215646,9,0,0,40,0,False,True,False,...,False,False,False,False,False,False,False,True,False,False
3,53,234721,7,0,0,40,0,False,True,False,...,False,False,False,False,False,False,False,True,False,False
4,28,338409,13,0,0,40,0,False,True,False,...,False,False,False,False,False,False,False,False,False,False


## 2. Train, test and validation split

In [2]:
X = df.drop(columns=["income"])
y = df["income"]

# 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# split the remaining 30% into 15% test / 15% validation
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f"Train set: {len(X_train)} rows ({len(X_train)/len(df):.1%})")
print(f"Test set:  {len(X_test)} rows ({len(X_test)/len(df):.1%})")
print(f"Val set:   {len(X_val)} rows ({len(X_val)/len(df):.1%})")

Train set: 21097 rows (70.0%)
Test set:  4521 rows (15.0%)
Val set:   4521 rows (15.0%)


## 3. K-means clustering

K-means uses distances, so the features are scaled with `StandardScaler` (fitted on the training set only). The `income` column is **not** a feature. K-means never sees it. First, a quick silhouette check over a few values of k.

In [3]:
scaler = StandardScaler().fit(X_train)
Xs_train, Xs_test, Xs_val = (scaler.transform(d) for d in (X_train, X_test, X_val))

for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs_train)
    sil = silhouette_score(Xs_train, km.labels_, sample_size=5000, random_state=42)
    print(f"k={k}: inertia {km.inertia_:.0f}, silhouette {sil:.4f}")

k=2: inertia 1948618, silhouette 0.0506


k=3: inertia 1910943, silhouette 0.0298


k=4: inertia 1891510, silhouette 0.0432


k=5: inertia 1864219, silhouette 0.0530


k=6: inertia 1834172, silhouette 0.0361


## 4. K-means with k = 2 compared with the previous results

Income has two classes, so k = 2. Each cluster gets the majority income label of the **training** rows in it. Then the test and validation rows are scored with that mapping.

**This is not a like-for-like comparison.** The supervised models learned from the income labels. K-means is unsupervised and never sees them.

In [4]:
km = KMeans(n_clusters=2, n_init=10, random_state=42).fit(Xs_train)
mapping = {c: int(y_train[km.labels_ == c].mean() >= 0.5) for c in range(2)}
print("Cluster sizes (train):", np.bincount(km.labels_).tolist())
print("Share of >50K per cluster (train):", y_train.groupby(km.labels_).mean().round(4).tolist())
print("Cluster -> income label:", mapping)

km_val = accuracy_score(y_val, pd.Series(km.predict(Xs_val)).map(mapping))
km_test = accuracy_score(y_test, pd.Series(km.predict(Xs_test)).map(mapping))
km_sil = silhouette_score(Xs_train, km.labels_, sample_size=5000, random_state=42)
print(f"k-means (k=2): validation {km_val:.2%}, test {km_test:.2%}, silhouette {km_sil:.4f}")
print(f"Majority class baseline (test): {max(y_test.mean(), 1 - y_test.mean()):.2%}")

previous = {
    "Decision Tree (Gini)": 81.20,
    "Decision Tree (Entropy)": 81.42,
    "Decision Tree (Entropy, max_depth=8, pruned)": 84.56,
    "SVM (RBF kernel)": 83.70,
    "KNN (k=5)": 81.53,
}
previous["K-means (k=2, unsupervised)"] = round(km_test * 100, 2)
table = pd.Series(previous, name="Test Accuracy (%)").sort_values(ascending=False).reset_index()
table.columns = ["Algorithm", "Test Accuracy (%)"]
table.index += 1
table

Cluster sizes (train): [10842, 10255]
Share of >50K per cluster (train): [0.0594, 0.4455]
Cluster -> income label: {0: 0, 1: 0}


k-means (k=2): validation 75.14%, test 74.14%, silhouette 0.0506
Majority class baseline (test): 74.14%


,Algorithm,Test Accuracy (%)
1,"Decision Tree (Entropy, max_depth=8, pruned)",84.56
2,SVM (RBF kernel),83.70
3,KNN (k=5),81.53
4,Decision Tree (Entropy),81.42
5,Decision Tree (Gini),81.20
6,"K-means (k=2, unsupervised)",74.14


## 5. Conclusions

- K-means (k=2) scores **74.14%** on the test set. That is the lowest in the table. It is 10.42 points below the best model (pruned Decision Tree, 84.56%) and 9.56 points below SVM (83.70%).
- Both clusters map to income label 0, because even the richer cluster has only 44.55% of rows above 50K (the other has 5.94%). So k-means predicts "<=50K" for everyone, and 74.14% equals the majority-class baseline.
- The clusters do carry some income signal (5.94% vs 44.55% above 50K). But they follow other structure in the 96 scaled features, and the silhouette score is low (0.0506), so the clusters are not well separated.
- This is not a like-for-like comparison. The supervised models used the income labels to learn. K-means never saw them. The result shows that income is not the main natural grouping in this data, not that k-means is a bad algorithm.
